# Day 8 | Lab 8.4: CrewAI — Role-Based Multi-Agent for Fintech Content

**Duration:** ~1.25 hours

**Scenario.** *BrightFin Media* — a fintech-focused content marketing agency in Mumbai serving 12 financial-services clients across India. The current Researcher → Writer → Editor → Compliance handoff chain runs on email and takes 3.5 days per post, with a 42% revision rate and 18% compliance flag rate. We replace the email chain with a CrewAI multi-agent pipeline.

**Learning Objectives.** By the end of this lab, you will be able to:
1. Define **CrewAI agents** with `role`, `goal`, and `backstory` — the framework's core mental model.
2. Define `Task` objects whose outputs flow as context to the next task in a `Crew`.
3. Run a `Crew` in `Process.sequential` mode (deterministic, fixed order).
4. Run a `Crew` in `Process.hierarchical` mode (manager LLM delegates dynamically).
5. Articulate when to reach for **CrewAI** (role-based content workflows) versus **LangGraph** (graph-based compliance / routing) versus the **OpenAI Agents SDK** (handoff-based, OpenAI-native).

**Why this lab matters for your client work.** Most enterprise content / research / drafting workflows today run as email chains between specialists. CrewAI gives you the cleanest mapping from that org chart onto a multi-agent system — each agent's `role` corresponds to a job title, `goal` to their KRA, `backstory` to their domain expertise. Once trainees see this mapping, they stop trying to model creative pipelines as state machines.

---


## 1. Install Dependencies


In [ ]:
# Required packages for this lab — already installed in your local venv.
# To install standalone, uncomment the line(s) below:
# !pip install -q 'crewai>=0.80' 'crewai-tools' 'langchain-openai>=1.0'


## 2. API Key Configuration

CrewAI uses **LiteLLM** under the hood, so any provider works. We default to OpenAI (`gpt-4o-mini`) here for speed and cost, but the same agents run unchanged on Anthropic or Groq if you swap the `llm=` string. Set keys in your local `.env`.


In [ ]:
import os

# Local-venv pattern: load from .env if python-dotenv is available, otherwise rely on
# environment variables already set in your shell or venv activation script.
try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

for key in ['OPENAI_API_KEY']:
    status = '✅ Loaded' if os.environ.get(key) else '❌ MISSING'
    print(f'{key}: {status}')


## 3. Imports


In [ ]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
import json

print('CrewAI loaded successfully!')
print('Core components: Agent, Task, Crew, Process, @tool')


## 4. Business Scenario — BrightFin Media (recap)

**Pain points the existing email pipeline causes:**

| Pain | Today | Target |
|---|---|---|
| Brief → publish turnaround | 3.5 days | < 4 hours |
| Major-revision rate | 42% | < 15% |
| Compliance flag rate | 18% | < 3% |
| Trend-lag on UPI / SIP topics | 2–3 days | same-day |

**Our pipeline (4 role-based agents):** Market Researcher → Content Writer → Editor → SEBI/RBI Compliance Reviewer. We will run the same crew in two modes — `sequential` (today's deterministic order) and `hierarchical` (a manager agent decides who does what next).


## 5. Understanding CrewAI — Role-Based Orchestration

CrewAI's philosophy: agents are **team members** with clear professional identities, *not* nodes in a graph and *not* anonymous handoff targets.

| Framework | Mental model | What you write |
|---|---|---|
| **LangGraph** | Build the workflow graph | nodes + edges + state |
| **OpenAI Agents SDK** | Define agents, let them hand off | `Agent(name, instructions, tools, handoffs)` |
| **CrewAI** | Hire a team, give them tasks | `Agent(role, goal, backstory)` + `Task(description, expected_output)` |

Two execution modes ship in the box:
- **`Process.sequential`** — tasks run in the order you list them; each task's output becomes the next task's context.
- **`Process.hierarchical`** — a manager LLM reads the goal, picks the next agent, and delegates dynamically.


## 6. Fintech Content Data

Inline data for BrightFin Media — content briefs, brand guidelines, compliance rules, and platform specs. All synthetic.


In [ ]:
# --- BrightFin Media: Inline Fintech Content Data ---

# Content briefs (what clients want published)
CONTENT_BRIEFS = [
    {
        "brief_id": "BF-001",
        "client": "PayRight (UPI Payments)",
        "topic": "UPI transactions crossing 15 billion/month milestone",
        "platform": "LinkedIn",
        "tone": "Professional, data-driven",
        "key_points": ["15B monthly transactions", "40% YoY growth", "India leading global real-time payments"],
        "cta": "Learn how PayRight processes 2M+ daily transactions"
    },
    {
        "brief_id": "BF-002",
        "client": "WealthFirst (Mutual Funds)",
        "topic": "SIP investments reaching all-time high of \u20b923,000 crore/month",
        "platform": "Twitter",
        "tone": "Conversational, engaging",
        "key_points": ["\u20b923K crore monthly SIP", "7.8 crore SIP accounts", "Power of rupee-cost averaging"],
        "cta": "Start your SIP journey with WealthFirst"
    },
    {
        "brief_id": "BF-003",
        "client": "SecureLend (Digital Lending)",
        "topic": "RBI's new digital lending guidelines and what they mean for borrowers",
        "platform": "LinkedIn",
        "tone": "Educational, authoritative",
        "key_points": ["Mandatory disclosure of all charges", "Cooling-off period for borrowers", "Data privacy requirements"],
        "cta": "SecureLend is 100% RBI-compliant. Check your eligibility."
    },
]

# Brand guidelines
BRAND_GUIDELINES = {
    "voice": "Confident, knowledgeable, approachable. Never condescending.",
    "forbidden_words": ["guaranteed returns", "risk-free", "100% safe", "get rich", "double your money"],
    "required_elements": ["SEBI/RBI disclaimer where applicable", "Data source attribution", "Client CTA"],
    "hashtag_rules": "Max 5 hashtags. Always include #Fintech #India. Platform-specific otherwise."
}

# Compliance rules (SEBI/RBI)
COMPLIANCE_RULES = {
    "mutual_funds": "Must include: 'Mutual fund investments are subject to market risks. Read all scheme-related documents carefully.'",
    "lending": "Must include: 'Terms and conditions apply. Loans subject to creditworthiness assessment.'",
    "payments": "No disclaimer required for factual payment statistics.",
    "general": "Never promise specific returns. Always attribute data sources. Include 'Past performance does not guarantee future results' for investment content."
}

# Platform specifications
PLATFORM_SPECS = {
    "LinkedIn": {"max_chars": 3000, "hashtags": 5, "tone": "Professional", "format": "Long-form with bullets"},
    "Twitter": {"max_chars": 280, "hashtags": 3, "tone": "Punchy", "format": "Thread-friendly, hook first"},
    "Instagram": {"max_chars": 2200, "hashtags": 15, "tone": "Visual, storytelling", "format": "Carousel-friendly"}
}

# Simulated market trends (what the research tool returns)
MARKET_TRENDS = {
    "UPI": ["UPI 3.0 features launching Q2 2026", "International UPI corridors expanding to 12 countries", "UPI Lite crosses 10M daily transactions"],
    "mutual_funds": ["SEBI simplifying KYC for mutual fund investments", "Passive index funds AUM crosses \u20b910 lakh crore", "NFO launches up 35% in 2026"],
    "digital_lending": ["RBI tightening FLDG norms for fintechs", "Digital lending market projected at $720B by 2030", "Co-lending partnerships growing 45% YoY"]
}

print(f"Data loaded: {len(CONTENT_BRIEFS)} briefs, {len(PLATFORM_SPECS)} platforms")
print(f"Sample brief: {CONTENT_BRIEFS[0]['client']} - {CONTENT_BRIEFS[0]['topic'][:50]}...")

## 7. Define Content-Pipeline Tools

CrewAI uses the `@tool` decorator (the same idea as LangChain's `@tool`). We define three tools that each agent can call:

1. `search_market_trends` — get latest fintech trends for a topic (simulated).
2. `check_compliance` — verify content meets SEBI/RBI guidelines.
3. `format_for_platform` — adapt content for a specific social platform.


In [ ]:
import json

# --- Tool 1: Market Trends ---
@tool("Search Market Trends")
def search_market_trends(topic: str) -> str:
    """Search for the latest fintech market trends related to a topic. Returns 3 trending insights."""
    topic_lower = topic.lower()
    for key, trends in MARKET_TRENDS.items():
        if key.lower() in topic_lower or topic_lower in key.lower():
            return f"Trending insights for '{topic}':\n" + "\n".join(f"  - {t}" for t in trends)
    return f"No specific trends found for '{topic}'. General fintech trends: Digital payments growing 28% YoY, embedded finance adoption accelerating."


# --- Tool 2: Compliance Check ---
@tool("Check Compliance")
def check_compliance(content: str, content_type: str) -> str:
    """Check if content meets SEBI/RBI compliance rules. content_type: mutual_funds, lending, payments, or general."""
    rule = COMPLIANCE_RULES.get(content_type.lower(), COMPLIANCE_RULES["general"])
    # Check for forbidden words
    violations = [w for w in BRAND_GUIDELINES["forbidden_words"] if w.lower() in content.lower()]
    if violations:
        return f"COMPLIANCE VIOLATION: Contains forbidden words: {violations}\nRequired disclaimer: {rule}"
    # Check for required disclaimer
    has_disclaimer = any(phrase in content.lower() for phrase in ["subject to market risks", "terms and conditions", "past performance"])
    if content_type in ["mutual_funds", "lending"] and not has_disclaimer:
        return f"COMPLIANCE WARNING: Missing required disclaimer.\nAdd: {rule}"
    return f"COMPLIANCE PASSED. Applicable rule: {rule}"


# --- Tool 3: Platform Formatter ---
@tool("Format for Platform")
def format_for_platform(content: str, platform: str) -> str:
    """Format content for a specific social media platform (LinkedIn, Twitter, or Instagram)."""
    spec = PLATFORM_SPECS.get(platform)
    if not spec:
        return f"Unknown platform '{platform}'. Supported: LinkedIn, Twitter, Instagram"
    char_count = len(content)
    status = "WITHIN LIMIT" if char_count <= spec["max_chars"] else f"OVER LIMIT by {char_count - spec['max_chars']} chars"
    return (
        f"Platform: {platform}\n"
        f"Character count: {char_count}/{spec['max_chars']} ({status})\n"
        f"Recommended tone: {spec['tone']}\n"
        f"Format: {spec['format']}\n"
        f"Max hashtags: {spec['hashtags']}"
    )


print("3 tools defined: search_market_trends, check_compliance, format_for_platform")

In [ ]:
# --- Test Tools ---
print("TEST: search_market_trends")
print(search_market_trends.run("UPI payments"))

print("\nTEST: check_compliance (should warn about missing disclaimer)")
print(check_compliance.run(content="SIP is the best way to invest. Start today!", content_type="mutual_funds"))

print("\nTEST: format_for_platform")
print(format_for_platform.run(content="A" * 300, platform="Twitter"))

## 8. Create the BrightFin Media Agents

Each agent has a **`role`** (job title), **`goal`** (what they're trying to achieve), and **`backstory`** (context that shapes their behaviour and tone). Tools are attached per-agent — only the Researcher needs `search_market_trends`, only the Editor needs `format_for_platform`, only the Compliance Reviewer needs `check_compliance`.


In [ ]:
# --- Agent 1: Market Researcher ---
researcher = Agent(
    role="Fintech Market Research Analyst",
    goal="Find the latest market trends, data points, and newsworthy angles for fintech content briefs",
    backstory=(
        "You are BrightFin Media's senior research analyst with 8 years of experience "
        "tracking Indian fintech markets. You follow NPCI, SEBI, and RBI announcements daily. "
        "You know the UPI ecosystem, mutual fund industry, and digital lending space inside out. "
        "Your research briefs are data-rich and always include specific numbers."
    ),
    tools=[search_market_trends],
    llm="openai/gpt-4o-mini",
    verbose=True,
)

print(f"Agent: {researcher.role}")
print(f"  Tools: {[t.name for t in researcher.tools]}")

In [ ]:
# --- Agent 2: Content Writer ---
writer = Agent(
    role="Fintech Content Writer",
    goal="Create engaging, platform-optimized social media content that educates and converts",
    backstory=(
        "You are BrightFin Media's lead content writer, specializing in making complex "
        "financial topics accessible to everyday Indians. You write for LinkedIn, Twitter, "
        "and Instagram. Your content balances educational value with strong CTAs. "
        "You never use jargon without explaining it and always write in an approachable tone."
    ),
    llm="openai/gpt-4o-mini",
    verbose=True,
)

print(f"Agent: {writer.role}")

In [ ]:
# --- Agent 3: Editor ---
editor = Agent(
    role="Content Editor & Quality Assurance",
    goal="Ensure content meets brand guidelines, platform specs, and editorial standards",
    backstory=(
        "You are BrightFin Media's chief editor with a keen eye for quality. "
        "You enforce brand voice consistency, check character limits, verify data accuracy, "
        "and ensure every post follows platform best practices. You're especially strict about "
        "the forbidden words list and hashtag guidelines."
    ),
    tools=[format_for_platform],
    llm="openai/gpt-4o-mini",
    verbose=True,
)

print(f"Agent: {editor.role}")

In [ ]:
# --- Agent 4: Compliance Reviewer ---
compliance_reviewer = Agent(
    role="SEBI/RBI Compliance Reviewer",
    goal="Ensure all fintech content complies with SEBI and RBI regulations",
    backstory=(
        "You are BrightFin Media's compliance specialist with deep knowledge of "
        "SEBI's advertising guidelines for mutual funds, RBI's digital lending norms, "
        "and general financial services advertising regulations in India. "
        "You flag any content that could mislead investors or violate regulatory norms. "
        "You add required disclaimers and remove forbidden claims."
    ),
    tools=[check_compliance],
    llm="openai/gpt-4o-mini",
    verbose=True,
)

print(f"Agent: {compliance_reviewer.role}")

## 9. Define Tasks for the Content Pipeline

Each `Task` says **what** needs to be done, **who** does it, and **what good output looks like**. In `Process.sequential`, each task's output is automatically threaded into the next task as context.


In [ ]:
# --- Select a content brief for processing ---
brief = CONTENT_BRIEFS[0]  # UPI post for LinkedIn
print(f"Processing brief: {brief['brief_id']} - {brief['client']}")
print(f"Topic: {brief['topic']}")
print(f"Platform: {brief['platform']} | Tone: {brief['tone']}")

# --- Task 1: Research ---
research_task = Task(
    description=(
        f"Research the latest trends and data points about: {brief['topic']}\n"
        f"Key points to cover: {', '.join(brief['key_points'])}\n"
        f"Use the search_market_trends tool to find current trends.\n"
        f"Provide a research brief with 5-7 data points and trend insights."
    ),
    agent=researcher,
    expected_output="A research brief with 5-7 data points, trend insights, and suggested angles for the content.",
)

# --- Task 2: Write ---
writing_task = Task(
    description=(
        f"Write a {brief['platform']} post about: {brief['topic']}\n"
        f"Tone: {brief['tone']}\n"
        f"CTA: {brief['cta']}\n"
        f"Use the research brief from the previous task as your data source.\n"
        f"Brand voice: {BRAND_GUIDELINES['voice']}\n"
        f"Forbidden words: {', '.join(BRAND_GUIDELINES['forbidden_words'])}"
    ),
    agent=writer,
    expected_output=f"A complete {brief['platform']} post ready for editorial review.",
)

# --- Task 3: Edit ---
editing_task = Task(
    description=(
        f"Review and edit the {brief['platform']} post for quality:\n"
        f"1. Check character count (max {PLATFORM_SPECS[brief['platform']]['max_chars']})\n"
        f"2. Verify brand voice consistency\n"
        f"3. Ensure hashtags follow guidelines (max {PLATFORM_SPECS[brief['platform']]['hashtags']})\n"
        f"4. Use the format_for_platform tool to validate formatting\n"
        f"5. Polish the writing for maximum engagement"
    ),
    agent=editor,
    expected_output="An edited, polished post that meets all platform and brand guidelines.",
)

# --- Task 4: Compliance ---
content_type = "payments"  # UPI content
compliance_task = Task(
    description=(
        f"Review the post for SEBI/RBI compliance:\n"
        f"Content type: {content_type}\n"
        f"1. Use the check_compliance tool to verify regulatory compliance\n"
        f"2. Check for forbidden claims (guaranteed returns, risk-free, etc.)\n"
        f"3. Add required disclaimers if needed\n"
        f"4. Verify data source attributions are present\n"
        f"Compliance rules: {COMPLIANCE_RULES.get(content_type, COMPLIANCE_RULES['general'])}"
    ),
    agent=compliance_reviewer,
    expected_output="Final compliance-approved post with all required disclaimers and a compliance status report.",
)

print("4 tasks defined: research -> writing -> editing -> compliance")

In [ ]:
# Print task summaries
for i, task in enumerate([research_task, writing_task, editing_task, compliance_task], 1):
    print(f"Task {i}: {task.agent.role}")
    print(f"  Expected: {task.expected_output[:80]}...")
    print()

## 10. Part 1 — Sequential Content Pipeline

In `Process.sequential`, tasks execute in declared order. This mirrors BrightFin's *current* email workflow exactly: Researcher → Writer → Editor → Compliance. The win is throughput and consistency, not creative re-routing.


In [ ]:
# --- Sequential Crew ---
sequential_crew = Crew(
    agents=[researcher, writer, editor, compliance_reviewer],
    tasks=[research_task, writing_task, editing_task, compliance_task],
    process=Process.sequential,
    verbose=True,
)

print("Sequential Crew assembled!")
print(f"  Agents: {[a.role for a in sequential_crew.agents]}")
print(f"  Process: Sequential (fixed order)")
print("\nKicking off the pipeline...")
print("=" * 60)

sequential_result = sequential_crew.kickoff()

print("\n" + "=" * 60)
print("SEQUENTIAL PIPELINE RESULT:")
print(sequential_result)

In [ ]:
# --- Analyze Sequential Pipeline ---
print("Sequential Pipeline Analysis")
print("=" * 60)
print(f"Brief: {brief['brief_id']} - {brief['client']}")
print(f"Platform: {brief['platform']}")
print(f"\nFinal output length: {len(str(sequential_result))} chars")
print(f"Pipeline flow: Researcher -> Writer -> Editor -> Compliance")
print(f"\nKey observation: Each agent built on the previous agent's output.")

## 11. Part 2 — Hierarchical Pipeline with a Manager Agent

In `Process.hierarchical`, a **manager LLM** reads the overall goal, looks at the available agents, and dynamically delegates tasks to whichever agent it judges best. *You* don't pick the order — the manager does.

Use this when the optimal order varies by brief (e.g., compliance-first for high-risk topics, research-first for trending news).


In [ ]:
# --- Use a different brief for hierarchical demo ---
brief_2 = CONTENT_BRIEFS[1]  # SIP/Mutual Funds post for Twitter
print(f"Processing brief: {brief_2['brief_id']} - {brief_2['client']}")
print(f"Topic: {brief_2['topic']}")

# Redefine tasks for brief_2
research_task_2 = Task(
    description=f"Research trends about: {brief_2['topic']}. Key points: {', '.join(brief_2['key_points'])}",
    agent=researcher,
    expected_output="Research brief with data points about SIP and mutual fund trends.",
)

writing_task_2 = Task(
    description=f"Write a {brief_2['platform']} post. Topic: {brief_2['topic']}. Tone: {brief_2['tone']}. CTA: {brief_2['cta']}",
    agent=writer,
    expected_output=f"A complete {brief_2['platform']} post (max {PLATFORM_SPECS[brief_2['platform']]['max_chars']} chars).",
)

editing_task_2 = Task(
    description=f"Edit the {brief_2['platform']} post. Check: character count (max 280), hashtags (max 3), tone (punchy).",
    agent=editor,
    expected_output="Polished tweet ready for compliance review.",
)

compliance_task_2 = Task(
    description=f"Verify SEBI compliance for mutual fund content. Required disclaimer: {COMPLIANCE_RULES['mutual_funds']}",
    agent=compliance_reviewer,
    expected_output="Compliance-approved tweet with required SEBI disclaimer.",
)

# --- Hierarchical Crew ---
hierarchical_crew = Crew(
    agents=[researcher, writer, editor, compliance_reviewer],
    tasks=[research_task_2, writing_task_2, editing_task_2, compliance_task_2],
    process=Process.hierarchical,
    manager_llm="openai/gpt-4o-mini",
    verbose=True,
)

print("\nHierarchical Crew assembled (manager will delegate)!")
print("=" * 60)

hierarchical_result = hierarchical_crew.kickoff()

print("\n" + "=" * 60)
print("HIERARCHICAL PIPELINE RESULT:")
print(hierarchical_result)

In [ ]:
# --- Compare Sequential vs Hierarchical ---
print("Sequential vs Hierarchical Comparison")
print("=" * 60)
print(f"\nSequential (LinkedIn post):")
print(f"  Flow: Fixed order (Research -> Write -> Edit -> Compliance)")
print(f"  Output length: {len(str(sequential_result))} chars")
print(f"  Control: You decide the order")

print(f"\nHierarchical (Twitter post):")
print(f"  Flow: Manager-delegated (manager decides order and assignments)")
print(f"  Output length: {len(str(hierarchical_result))} chars")
print(f"  Control: Manager LLM decides the order")

print(f"\nKey Difference:")
print(f"  Sequential = predictable, consistent (like a factory assembly line)")
print(f"  Hierarchical = flexible, adaptive (like a project manager delegating)")

## 12. CrewAI vs LangGraph vs OpenAI Agents SDK

Now that we've used three frameworks across Module 7 (LangGraph) and Module 8 (Supervisor / Orchestrator-Worker, MAF, OpenAI SDK in 8.5, and CrewAI here), here is the side-by-side cheat-sheet:

| Dimension | LangGraph | OpenAI Agents SDK | CrewAI |
|---|---|---|---|
| **Philosophy** | Build the workflow graph | Define agents, let them hand off | Hire a team with roles |
| **Orchestration** | Graph (nodes + edges) | Handoffs (agent → agent) | Role-based (sequential / hierarchical) |
| **Control** | Maximum — every path explicit | Medium — LLM picks handoffs | Medium — `Process` type decides |
| **Agent Definition** | `create_agent(tools, system_prompt)` | `Agent(name, instructions, tools)` | `Agent(role, goal, backstory, tools)` |
| **Task Concept** | Implicit (nodes do work) | Implicit (agent responds) | **Explicit** `Task(description, expected_output)` |
| **State** | Explicit `TypedDict` | Conversation history | Task-context chain |
| **Multi-Agent** | Subgraphs, Supervisor, Send | Handoffs, Agents-as-Tools | Sequential, Hierarchical |
| **Guardrails** | Custom / external | Built-in Input/Output | Callbacks, validation |
| **Memory** | `MemorySaver` + `InMemoryStore` | `SQLiteSession` | Crew memory (built-in) |
| **LLM Providers** | Any | OpenAI only | Any (via LiteLLM) |
| **Best For** | Complex enterprise workflows, compliance routing | OpenAI-native, voice, quick setup | Content / research / creative teams |


## 13. Why CrewAI for Content Pipelines but LangGraph for Compliance Workflows

This is the framework-selection mental model trainees need to take to a client interview.

### Reach for **CrewAI** when…
- The workflow already maps cleanly onto an **org chart** — Researcher, Writer, Editor, Reviewer, etc.
- Each step's success criterion is **qualitative** — "a good post", "compliant copy", "well-edited draft" — and benefits from an LLM agent with a domain backstory.
- Order is **mostly fixed** with occasional manager-driven re-routing (sequential / hierarchical processes).
- You want **fast time-to-prototype** — a 4-agent crew is ~30 lines of code.
- Examples: content marketing pipelines, research syntheses, RFP drafting, due-diligence write-ups, internal newsletter generation.

### Reach for **LangGraph** when…
- The workflow is a **state machine** — explicit branches, retries, conditional edges, parallel fan-out, human-in-the-loop interrupts mid-graph.
- Success is **deterministic and auditable** — "if amount > ₹2L AND no KYC, route to manual review" — and the regulator will ask to see the routing logic.
- You need to **inspect / replay** state at every step (LangSmith traces, checkpointers).
- You may need to **interrupt** for human approval (e.g., HITL on the compliance step) — `interrupt()` + `Command(resume=...)` is first-class.
- Examples: compliance report generation (Lab 8.2), payment-routing engines, KYC orchestration, escalation workflows, anything that has to pass an audit.

### Reach for the **OpenAI Agents SDK** when…
- The stack is OpenAI-native and you want **built-in input/output guardrails + tracing** without wiring your own.
- The control flow is well captured by **handoffs** (concierge → specialist) rather than explicit graphs or fixed task sequences.
- You need **voice** (`RealtimeAgent`) — no other framework on this list ships that.
- See Lab 8.5 for the full hands-on.

### One-line decision rule
> **Org chart → CrewAI · State machine → LangGraph · Concierge with handoffs → OpenAI Agents SDK · Open-source on Azure → Microsoft Agent Framework (Lab 8.3).**


## 14. Conclusion & Key Takeaways

In this lab we built BrightFin Media's content pipeline with CrewAI:

| Concept | What we used |
|---|---|
| Role-based agents | `Agent(role, goal, backstory, tools, llm)` × 4 |
| Tools | `@tool`-decorated callables shared across agents |
| Tasks | `Task(description, agent, expected_output)` × 4 |
| Sequential process | `Crew(..., process=Process.sequential).kickoff()` |
| Hierarchical process | `Crew(..., process=Process.hierarchical, manager_llm=...).kickoff()` |
| Framework selection | CrewAI for content / org-chart workflows; LangGraph for compliance state machines |

**Why role / goal / backstory matters.** A well-written backstory acts like a permanent, role-scoped system prompt that nudges the agent's tone, vocabulary, and risk appetite *before* any task ever arrives. Investing 60 seconds in a good backstory often beats hours of task-prompt tuning.

### Next Lab Preview
**Lab 8.5 — OpenAI Agents SDK** — we rebuild a similar multi-agent system (banking concierge with 3 specialists) using OpenAI's handoff-based SDK and add **input/output guardrails** + **session memory** + **built-in tracing**.


## 15. Stretch Exercise (Optional)

Pick **one** of the following — each takes ~20 minutes:

1. **Add a 5th agent — Performance Analyst.** Give it a tool `predict_engagement(post: str, platform: str)` that returns a fake engagement score (e.g., based on word count + emoji count + hashtag count). Add a 5th task that runs after compliance and recommends tweaks if the score is < 70. Re-run the sequential crew and compare outputs.

2. **Swap the LLM provider per agent.** Keep the Researcher on `openai/gpt-4o-mini` (cheap, factual), but give the Writer `anthropic/claude-haiku-4-5` (better tone for marketing copy). Confirm both providers work in the same crew.

3. **Force a compliance failure and observe.** Manually inject a forbidden phrase like *"guaranteed returns"* into the writing-task description for `brief_2` and re-run. Read the verbose trace and confirm the Compliance Reviewer catches it and asks the Writer to revise.

4. **Persist the crew output.** After both `kickoff()` calls, write `sequential_result` and `hierarchical_result` to `outputs/lab_8_4_brightfin_pipeline.json` with timestamp + brief metadata. This is the basis of an audit log for the marketing team.
